In [1]:
import os
import sys
import json
import pickle
import random
from pathlib import Path
import numpy as np
import torch

# 切换工作目录
os.chdir(r"E:\student\模型")
print(f"当前工作目录: {os.getcwd()}")

# 添加项目路径
sys.path.insert(0, r"E:\student\模型")

from traffic_detector.config import CHECKPOINT_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR
from traffic_detector.models.flow_mlp import FlowMLPDetector

CHECKPOINT_PATH = CHECKPOINT_DIR / "flow_mlp_best.pt"
CONFIG_PATH = CHECKPOINT_DIR / "flow_mlp_config.json"
SCALER_PATH = PROCESSED_DATA_DIR / "scaler.pkl"
DEMO_DATA_PATH = PROCESSED_DATA_DIR / "demo_samples.pkl"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"配置完成，使用设备: {DEVICE}")

当前工作目录: E:\student\模型
配置完成，使用设备: cuda


In [2]:


from traffic_detector.features.flow_schema import get_default_flow_feature_spec
from traffic_detector.features.flow_transform import FlowFeatureTransformer

print("正在准备演示数据（这需要1-2分钟）...")

feature_spec = get_default_flow_feature_spec()
transformer = FlowFeatureTransformer(feature_spec=feature_spec)

attack_path = RAW_DATA_DIR / "dataset_attack.csv"
normal_path = RAW_DATA_DIR / "dataset_normal.csv"

X, y = transformer.fit_transform_from_csv(str(attack_path), str(normal_path))

# 分层抽样
malicious_indices = np.where(y == 1)[0]
normal_indices = np.where(y == 0)[0]

num_malicious = min(50, len(malicious_indices))
num_normal = min(50, len(normal_indices))

selected_malicious = random.sample(list(malicious_indices), num_malicious)
selected_normal = random.sample(list(normal_indices), num_normal)
selected_indices = selected_malicious + selected_normal
random.shuffle(selected_indices)

X_demo = X[selected_indices]
y_demo = y[selected_indices]

# 保存
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(DEMO_DATA_PATH, 'wb') as f:
    pickle.dump({
        'X': X_demo,
        'y': y_demo,
        'feature_names': feature_spec.feature_names
    }, f)

print(f"已准备 {len(X_demo)} 个演示样本并保存")

正在准备演示数据（这需要1-2分钟）...
[fit_transform] read csv done: attack_shape=(2990062, 29), normal_shape=(2668936, 29)
[fit_transform] merged shape=(5658998, 30), building flow features...
[fit_transform] flow features ready, columns=42
[fit_transform] mapped to matrix: X_raw shape=(5658998, 38)
[fit_transform] scaling done
已准备 100 个演示样本并保存


In [3]:
# 载入配置
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = json.load(f)

print(f" 配置载入: 特征维度 {len(config['feature_names'])}, 隐藏层 {config['hidden_dims']}")

# 载入模型
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model = FlowMLPDetector(
    input_dim=checkpoint['input_dim'],
    hidden_dims=checkpoint['hidden_dims']
)
model.load_state_dict(checkpoint['model_state'])
model.to(DEVICE)
model.eval()

print(f"模型载入成功，参数量: {sum(p.numel() for p in model.parameters()):,}")

# 载入 scaler
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

print(" Scaler 载入成功")

# 载入演示数据
with open(DEMO_DATA_PATH, 'rb') as f:
    demo_data = pickle.load(f)

X_all = demo_data['X']
y_all = demo_data['y']
feature_names = demo_data['feature_names']

print(f"演示数据载入成功，共 {len(X_all)} 个样本")

 配置载入: 特征维度 38, 隐藏层 [256, 256, 128, 64]
模型载入成功，参数量: 184,705
 Scaler 载入成功
演示数据载入成功，共 100 个样本


In [ ]:
# 随机抽取样本进行演示
NUM_SAMPLES = 10
indices = random.sample(range(len(X_all)), min(NUM_SAMPLES, len(X_all)))

results = []

for idx in indices:
    features = X_all[idx]
    true_label = y_all[idx]
    
    # 标准化并预测
    features_scaled = scaler.transform(features.reshape(1, -1))
    features_tensor = torch.from_numpy(features_scaled).float().to(DEVICE)
    
    with torch.no_grad():
        logits, _ = model(features_tensor, return_logits=True)
        prob = torch.sigmoid(logits).cpu().item()
    
    pred_label = 1 if prob >= 0.5 else 0
    
    # 获取Top特征
    abs_features = np.abs(features)
    top_indices = np.argsort(abs_features)[-5:][::-1]
    top_features = [(feature_names[i], features[i]) for i in top_indices]
    
    results.append({
        'true_label': true_label,
        'pred_label': pred_label,
        'confidence': prob if pred_label == 1 else (1 - prob),
        'top_features': top_features
    })

print(f" 完成 {len(results)} 个样本的预测")

In [ ]:
# 显示预测结果
correct = 0

for i, result in enumerate(results):
    print("=" * 80)
    print(f"样本 #{i+1}")
    print("=" * 80)
    
    true_text = " 恶意流量" if result['true_label'] == 1 else " 正常流量"
    pred_text = " 恶意流量" if result['pred_label'] == 1 else " 正常流量"
    is_correct = result['pred_label'] == result['true_label']
    correct += is_correct
    
    print(f"\n 预测: {pred_text} (置信度: {result['confidence']:.2%}) {'✅' if is_correct else '❌'}")
    print(f"   真实: {true_text}")
    
    print(f"\n 关键特征 (Top 5):")
    for name, value in result['top_features']:
        print(f"   {name}: {value:.4f}")
    print()

print("=" * 80)
print(f" 统计: 准确率 {correct}/{len(results)} = {correct/len(results):.2%}")
print("=" * 80)